# WFI2033 data_visullization

读取 WFI2033 HMC 结果并可视化：
- Trace plot
- Corner plot
- Data / Model / Residual / Source（含 source-plane AGN trace 点）
- Base PSF 与 corrected PSF

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import sys

import arviz as az
import corner
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import numpyro
from astropy.io import fits

jax.config.update("jax_enable_x64", True)
numpyro.enable_x64()

# Ensure local WFI2033 modules are importable
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

from herculens_import_main import get_pixel_grid, trace_current_agn_to_source
from herculens.Instrument.noise import Noise
from herculens.Instrument.psf import PSF
from herculens.MassModel.mass_model import MassModel
from herculens.LightModel.light_model import LightModel
from herculens.PointSourceModel.point_source_model import PointSourceModel
from lens_images_extension import LensImageExtension, pixelize_plane as pixelize_plane_single

In [ ]:
pix_scale = 0.031  # arcsec / pixel

DATA_DIR = Path("../../Data/WFI2033")
RAW_DATA_PATH = DATA_DIR / "jw01198-o004_t004_nircam_clear-f115w_i2d.fits"
MASK_PATH = DATA_DIR / "mask_out_center.fits"
MASK_OUT_PATH = DATA_DIR / "mask_out_center.fits"

products_dir = Path('./result/read_data/data_products')
NC_PATH = Path('/mnt/lustre/tianli/quasar_hmc/WFI2033_psf_correct_all.nc')

with fits.open(RAW_DATA_PATH, memmap=True) as hdul_raw:
    exposure_time = float(hdul_raw[0].header.get("EXPTIME", hdul_raw[0].header.get("TEXPTIME", 1.0)))

data = np.array(fits.getdata(products_dir / 'data_bkg_sub.fits'), dtype=float)
rms_file = np.array(fits.getdata(products_dir / 'rms_with_psf_extra.fits'), dtype=float)
psf_base = np.array(fits.getdata(products_dir / 'psf_base.fits'), dtype=float)
mask = np.array(fits.getdata(MASK_PATH), dtype=bool)
mask_out = np.array(fits.getdata(MASK_OUT_PATH), dtype=bool)
mask = mask_out.copy()

psf_base = np.clip(psf_base, 0.0, None)
psf_base = psf_base / np.sum(psf_base)

inf_data_pixel = az.from_netcdf(str(NC_PATH))
post = inf_data_pixel.posterior
num_chains = int(post.sizes["chain"])

conj_points = jnp.array([
    [1.20212170716053, -0.12271885209256231],
    [0.9053233071260114, 0.5277189685977776],
    [-1.0461673774453952, 1.0081083299749878],
    [-0.1255456241215261, -0.8965524340129204],
])

G1_MASS_CENTER = (1.556, 1.299)
G2_MASS_CENTER = (2.145, -3.326)

pixel_grid, xgrid, ygrid, x_axis, y_axis, extent, nx, ny = get_pixel_grid(jnp.array(data), pix_scale)


In [ ]:
# Build LensImage object (pixelated source) for forward model visualization
noise = Noise(nx, ny, exposure_time=exposure_time)
psf_obj = PSF(psf_type='PIXEL', kernel_point_source=psf_base)

mass_model_pixel = MassModel(['EPL', 'SHEAR', 'SIS', 'SIS'])
lens_light_model_pixel = LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {})

pixel_grid_shape = int(post['pixels_source_grid'].shape[-1])

source_light_model_pixel = LightModel(
    ['PIXELATED'],
    pixel_adaptive_grid=True,
    pixel_interpol='fast_bilinear',
    kwargs_pixelated={'num_pixels': pixel_grid_shape},
)

point_source_model_pixel = PointSourceModel(
    ['IMAGE_POSITIONS'],
    mass_model=mass_model_pixel,
    image_plane=pixel_grid,
)

lens_image_pixel = LensImageExtension(
    pixel_grid,
    psf_obj,
    noise_class=noise,
    lens_light_model_class=lens_light_model_pixel,
    lens_mass_model_class=mass_model_pixel,
    source_model_class=source_light_model_pixel,
    point_source_model_class=point_source_model_pixel,
    source_arc_mask=jnp.array(mask),
    conjugate_points=conj_points,
    kwargs_numerics={'supersampling_factor': 1},
    source_grid_scale=1,
)


In [ ]:
def post_median(name: str):
    return np.array(post[name].median(dim='draw').values)


def first_scalar(x):
    a = np.asarray(x)
    return float(a.reshape(-1)[0])


model_image_med = post_median('model_image')
psf_corr_med = post_median('psf_kernel_corrected')
psf_corr_med = np.clip(psf_corr_med, 0.0, None)
psf_corr_med = psf_corr_med / np.sum(psf_corr_med, axis=(1, 2), keepdims=True)

theta_E_1_med = post_median('theta_E_1')
gamma_1_med = post_median('gamma_1')
e_1_med = post_median('e_1')
center_1_med = post_median('center_1')
theta_E_g1_med = post_median('theta_E_g1')
theta_E_g2_med = post_median('theta_E_g2')

sigma_lens_med = post_median('sigma_lens')
amp_lens_med = post_median('amp_lens')
e_lens_med = post_median('e_lens')
center_lens_med = post_median('center_lens')
pixels_source_med = post_median('pixels_source_grid')
ra_ps_med = post_median('ra_ps')
dec_ps_med = post_median('dec_ps')
log10_amp_ps_med = post_median('log10_amp_ps')

kwargs_chains = []
for i in range(num_chains):
    center_1_i = np.asarray(center_1_med[i])
    e_1_i = np.asarray(e_1_med[i])
    shear_i = np.asarray(post['gamma_sheer_1'].median(dim='draw').values[i])

    kwargs_lens = [
        {
            'theta_E': first_scalar(theta_E_1_med[i]),
            'gamma': first_scalar(gamma_1_med[i]),
            'e1': first_scalar(e_1_i[0]),
            'e2': first_scalar(e_1_i[1]),
            'center_x': float(center_1_i[0]),
            'center_y': float(center_1_i[1]),
        },
        {
            'gamma1': first_scalar(shear_i[0]),
            'gamma2': first_scalar(shear_i[1]),
            'ra_0': float(center_1_i[0]),
            'dec_0': float(center_1_i[1]),
        },
        {
            'theta_E': first_scalar(theta_E_g1_med[i]),
            'center_x': float(G1_MASS_CENTER[0]),
            'center_y': float(G1_MASS_CENTER[1]),
        },
        {
            'theta_E': first_scalar(theta_E_g2_med[i]),
            'center_x': float(G2_MASS_CENTER[0]),
            'center_y': float(G2_MASS_CENTER[1]),
        },
    ]

    e_l_i = np.asarray(e_lens_med[i])
    c_l_i = np.asarray(center_lens_med[i])
    kwargs_lens_light = [{
        'amp': np.asarray(amp_lens_med[i]),
        'sigma': np.asarray(sigma_lens_med[i]),
        'e1': np.asarray(e_l_i[0]),
        'e2': np.asarray(e_l_i[1]),
        'center_x': np.asarray(c_l_i[0]),
        'center_y': np.asarray(c_l_i[1]),
    }]

    kwargs_source = [{'pixels': np.asarray(pixels_source_med[i])}]

    kwargs_point_source = [{
        'ra': np.asarray(ra_ps_med[i]),
        'dec': np.asarray(dec_ps_med[i]),
        'amp': np.power(10.0, np.asarray(log10_amp_ps_med[i])),
    }]

    kwargs_chains.append({
        'kwargs_lens': kwargs_lens,
        'kwargs_lens_light': kwargs_lens_light,
        'kwargs_source': kwargs_source,
        'kwargs_point_source': kwargs_point_source,
    })


In [ ]:
# Trace + corner diagnostics
vars_mass = ['theta_E_1', 'theta_E_g1', 'theta_E_g2', 'gamma_1', 'e_1', 'center_1', 'gamma_sheer_1']
vars_power = ['n_source_grid', 'rho_source_grid', 'sigma_source_grid']

plt.rcParams['figure.constrained_layout.use'] = True
_ = az.plot_trace(
    inf_data_pixel.sel(chain=np.arange(num_chains)),
    var_names=vars_mass + vars_power,
    figsize=(12, 14),
)
plt.show()

fig_corner = None
for i in range(num_chains):
    fig_corner = corner.corner(
        inf_data_pixel.posterior.isel(chain=i),
        var_names=vars_mass,
        color=f'C{i}',
        fig=fig_corner,
    )
plt.show()


In [ ]:
# Main visualization: data/model/residual/source(+conjugate points) + PSF/base-vs-corrected
mask_plot = np.array(mask_out, dtype=bool)
rms_safe = np.array(rms_file, dtype=float)

for i in range(num_chains):
    kwargs_i = kwargs_chains[i]
    psf_corr_i = np.array(psf_corr_med[i], dtype=float)

    model_i = np.array(
        lens_image_pixel.model(
            **kwargs_i,
            source_add=True,
            point_source_add=True,
            psf_kernel=psf_corr_i,
        )
    )

    model_det_i = np.array(model_image_med[i], dtype=float)
    residual_i = (data - model_i) / rms_safe

    lens_light_i = np.array(
        lens_image_pixel.model(
            **kwargs_i,
            source_add=False,
            point_source_add=True,
            psf_kernel=psf_corr_i,
        )
    )

    source_i, source_extent = pixelize_plane_single(
        lens_image_pixel,
        kwargs_i,
        pixel_grid_shape,
    )
    src_x, src_y = trace_current_agn_to_source(lens_image_pixel, kwargs_i)
    src_x = np.array(src_x)
    src_y = np.array(src_y)

    fig, ax = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f'WFI2033 posterior median visualization | chain {i}', y=0.98)

    ax[0, 0].imshow(np.ma.array(data, mask=~mask_plot), norm='log', cmap='twilight', origin='lower', extent=extent)
    ax[0, 0].set_title('data')

    ax[0, 1].imshow(np.ma.array(model_i, mask=~mask_plot), norm='log', cmap='twilight', origin='lower', extent=extent)
    ax[0, 1].set_title('model (lens_image)')

    im_res = ax[0, 2].imshow(np.ma.array(residual_i, mask=~mask_plot), cmap='bwr', vmin=-3, vmax=3, origin='lower', extent=extent)
    ax[0, 2].set_title('residual / rms')
    plt.colorbar(im_res, ax=ax[0, 2], fraction=0.046, pad=0.04)

    ax[0, 3].imshow(np.ma.array(model_i - lens_light_i, mask=~mask_plot), norm='log', cmap='twilight', origin='lower', extent=extent)
    ax[0, 3].set_title('source + point source')

    src_abs = np.nanmax(np.abs(np.array(source_i)))
    src_norm = colors.SymLogNorm(
        linthresh=max(src_abs * 1e-3, 1e-8),
        vmin=-src_abs,
        vmax=src_abs,
    )
    ax[1, 0].imshow(np.array(source_i), cmap='twilight', norm=src_norm, origin='lower', extent=source_extent)
    ax[1, 0].scatter(src_x, src_y, s=90, facecolors='none', edgecolors='cyan', linewidths=1.6)
    for j, (sx, sy) in enumerate(zip(src_x, src_y), start=1):
        ax[1, 0].text(sx + 0.01, sy + 0.01, str(j), color='cyan', fontsize=10, weight='bold')
    ax[1, 0].set_title('source model + traced AGN points')

    eps = 1e-12
    vmax_psf = np.nanmax([np.nanmax(psf_base), np.nanmax(psf_corr_i)])
    psf_norm = colors.LogNorm(vmin=max(vmax_psf * 1e-6, eps), vmax=max(vmax_psf, eps * 10))

    ax[1, 1].imshow(np.clip(psf_base, eps, None), cmap='viridis', norm=psf_norm, origin='lower')
    ax[1, 1].set_title('PSF base')

    ax[1, 2].imshow(np.clip(psf_corr_i, eps, None), cmap='viridis', norm=psf_norm, origin='lower')
    ax[1, 2].set_title('PSF corrected')

    ax[1, 3].imshow(np.ma.array(model_det_i, mask=~mask_plot), norm='log', cmap='twilight', origin='lower', extent=extent)
    ax[1, 3].set_title('model_image (posterior det.)')

    for a in ax.ravel():
        a.set_xticks([])
        a.set_yticks([])

    plt.show()
